# LAB 05 — K-Means: khám phá các nhóm sinh viên

**DNU Learning Analytics Lab**

- **Họ tên:** ...
- **MSSV:** ...
- **Lớp:** ...

> **Câu hỏi trung tâm:** Nếu không dùng nhãn `Needs_Support`, dữ liệu học tập có tự cho thấy những nhóm sinh viên có đặc điểm tương đồng hay không?


## Environment checkpoint

Trước khi bắt đầu, chạy `python scripts/download_data.py` từ thư mục gốc repo. Sau đó chạy cell dưới đây.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

from kmeans_manual import init_centroids, assign_clusters, update_centroids, kmeans_manual

print('Environment ready!')


# Mission 1 — K-Means from scratch

K-Means lặp lại hai thao tác cốt lõi: **Assign → Update**.

## Nhiệm vụ

1. Mở `kmeans_manual.py`.
2. Hoàn thiện 4 hàm TODO.
3. Chạy `python -m pytest -q tests/test_kmeans_manual.py`.
4. Khi test pass, quay lại notebook và chạy ví dụ nhỏ dưới đây.

Không dùng `sklearn` để hoàn thiện phần này.


In [ ]:
toy = np.array([
    [5, 65],
    [7, 68],
    [18, 85],
    [20, 88],
    [30, 95],
    [28, 92],
], dtype=float)

centroids, labels, n_iters = kmeans_manual(toy, k=3, random_state=42)

print('Centroids:')
print(centroids)
print('Labels:', labels)
print('Iterations:', n_iters)


In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(toy[:, 0], toy[:, 1], c=labels, s=80)
plt.scatter(centroids[:, 0], centroids[:, 1], marker='X', s=220, edgecolor='black')
plt.xlabel('Hours_Studied')
plt.ylabel('Attendance')
plt.title('Toy K-Means result')
plt.show()


### Kết luận Mission 1

- Trong bước **Assign**, thuật toán làm gì? **[VIẾT]**
- Trong bước **Update**, centroid mới được tính thế nào? **[VIẾT]**
- Điều kiện dừng của phiên bản bạn cài là gì? **[VIẾT]**


# Mission 2 — Back to DNU case study

Ta quay lại **Student Performance Factors**.

Trong Lab 05, mục tiêu không phải dự đoán một nhãn có sẵn mà là **khám phá cấu trúc ẩn** trong các đặc trưng học tập.


In [ ]:
df = pd.read_csv('data/StudentPerformanceFactors.csv')

# Nhãn giảng dạy dùng lại từ chuỗi case study trước — KHÔNG dùng để huấn luyện K-Means.
df['Needs_Support'] = (df['Exam_Score'] < 65).astype(int)

feature_cols = [
    'Hours_Studied',
    'Attendance',
    'Previous_Scores',
    'Sleep_Hours',
    'Tutoring_Sessions',
]

display(df[feature_cols + ['Exam_Score', 'Needs_Support']].head())
print('Shape:', df.shape)
print('Missing in clustering features:')
display(df[feature_cols].isna().sum())


### Câu hỏi Mission 2

1. Tại sao `Needs_Support` không được đưa vào `feature_cols`? **[VIẾT]**
2. Tại sao `Exam_Score` cũng được giữ ngoài feature set chính của bài clustering này? **[VIẾT]**
3. Tại sao ta chưa đưa các biến categorical như `Gender`, `Motivation_Level` vào K-Means cơ bản? **[VIẾT]**


# Mission 3 — Scaling matters

K-Means dựa vào khoảng cách. Các biến có thang đo khác nhau có thể ảnh hưởng không cân bằng tới khoảng cách.

## Nhiệm vụ

1. Tạo `X` từ 5 feature đã chọn.
2. Chuẩn hóa bằng `StandardScaler`.
3. Chạy K-Means với cùng `K=3` trên dữ liệu gốc và dữ liệu scaled.
4. So sánh kết quả phân cụm.


In [ ]:
X = df[feature_cols].copy()

if X.isna().any().any():
    raise ValueError('Clustering features contain missing values. Hãy xử lý trước khi tiếp tục.')

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

raw_model = KMeans(n_clusters=3, random_state=42, n_init='auto')
scaled_model = KMeans(n_clusters=3, random_state=42, n_init='auto')

raw_labels = raw_model.fit_predict(X)
scaled_labels = scaled_model.fit_predict(X_scaled)

print('Cluster sizes — raw')
print(pd.Series(raw_labels).value_counts().sort_index())
print('
Cluster sizes — scaled')
print(pd.Series(scaled_labels).value_counts().sort_index())

print('
Cross-tab raw vs scaled cluster assignments')
display(pd.crosstab(raw_labels, scaled_labels, rownames=['Raw'], colnames=['Scaled']))


### Kết luận Mission 3

- Kết quả trước và sau scaling có giống nhau hoàn toàn không? **[VIẾT]**
- Biến nào có thang đo lớn hơn và có thể chi phối khoảng cách nếu không scale? **[VIẾT]**
- Vì sao scaling là bước đặc biệt quan trọng với K-Means? **[VIẾT]**


# Mission 4 — Choose K: Elbow + Silhouette

Không mặc định `K=3`. Hãy thử `K = 2..8`.

Với mỗi K:

- fit K-Means trên `X_scaled`;
- lưu `inertia_`;
- tính `silhouette_score`.


In [ ]:
ks = range(2, 9)
inertias = []
silhouettes = []

for k in ks:
    # TODO: khởi tạo và fit KMeans với random_state=42, n_init='auto'
    # TODO: append inertia vào inertias
    # TODO: tính silhouette_score và append vào silhouettes
    pass


In [ ]:
# TODO: chạy cell này sau khi hoàn thiện vòng lặp phía trên
assert len(inertias) == len(list(ks)), 'Bạn chưa tính đủ inertia cho K=2..8.'
assert len(silhouettes) == len(list(ks)), 'Bạn chưa tính đủ silhouette cho K=2..8.'

plt.figure(figsize=(7, 4))
plt.plot(list(ks), inertias, marker='o')
plt.xlabel('K')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(list(ks), silhouettes, marker='o')
plt.xlabel('K')
plt.ylabel('Silhouette Score')
plt.title('Silhouette by K')
plt.show()


### Quyết định Mission 4

- Elbow gợi ý K khoảng: **[ĐIỀN]**
- Silhouette cao nhất tại K = **[ĐIỀN]**
- Tôi chọn **K = [ĐIỀN]** cho mô hình cuối.

**Lý do:**  
[VIẾT 4–6 câu. Không chỉ ghi “vì điểm cao nhất”. Hãy cân nhắc cả khả năng diễn giải.]


# Mission 5 — Final K-Means model

Điền K mà bạn đã chọn ở Mission 4. Huấn luyện mô hình cuối trên **dữ liệu đã scale**.


In [ ]:
# TODO: thay None bằng K bạn chọn
best_k = None

assert isinstance(best_k, int) and 2 <= best_k <= 8, 'Hãy đặt best_k là một số nguyên từ 2 đến 8.'

final_model = KMeans(n_clusters=best_k, random_state=42, n_init='auto')
final_labels = final_model.fit_predict(X_scaled)

analysis_df = df.loc[X.index].copy()
analysis_df['Cluster'] = final_labels

print('Cluster sizes:')
display(analysis_df['Cluster'].value_counts().sort_index())


### Câu hỏi Mission 5

1. Các cluster có kích thước tương đối cân bằng không? **[VIẾT]**
2. Có cluster nào rất nhỏ không? Nếu có, điều đó gợi ý điều gì cần kiểm tra thêm? **[VIẾT]**


# Mission 6 — Cluster profiling

Cluster ID chỉ là số. Muốn hiểu ý nghĩa, ta phải xem **profile** của từng cluster.


In [ ]:
profile = analysis_df.groupby('Cluster')[feature_cols].mean()
display(profile.round(2))

scaled_profile = pd.DataFrame(X_scaled, columns=feature_cols, index=X.index)
scaled_profile['Cluster'] = final_labels
scaled_profile = scaled_profile.groupby('Cluster')[feature_cols].mean()

plt.figure(figsize=(9, 4.5))
sns.heatmap(scaled_profile, annot=True, fmt='.2f', center=0, cmap='vlag')
plt.title('Cluster profile on standardized features')
plt.show()


### Diễn giải Mission 6

Hoàn thiện bảng bằng bằng chứng từ profile/heatmap. Dùng tên mô tả trung tính.

| Cluster | Đặc điểm nổi bật | Tên mô tả đề xuất | Bằng chứng |
|---|---|---|---|
| 0 | [VIẾT] | [VIẾT] | [VIẾT] |
| 1 | [VIẾT] | [VIẾT] | [VIẾT] |
| ... | ... | ... | ... |

> **Cluster label ≠ Meaning.** K-Means không tự nói nhóm nào “tốt” hay “xấu”.


# Mission 7 — Post-hoc analysis: cluster có liên hệ gì với kết quả học tập?

Bây giờ mới dùng `Needs_Support` và `Exam_Score` để **kiểm tra sau phân cụm**. Không retrain K-Means.


In [ ]:
support_rate = pd.crosstab(
    analysis_df['Cluster'],
    analysis_df['Needs_Support'],
    normalize='index'
)

print('Needs_Support distribution by cluster')
display(support_rate.round(3))

print('Exam score by cluster')
display(analysis_df.groupby('Cluster')['Exam_Score'].agg(['mean', 'median', 'std']).round(2))


### Kết luận Mission 7

1. Cluster nào có tỷ lệ `Needs_Support = 1` cao hơn? **[VIẾT]**
2. Cluster nào có `Exam_Score` trung bình thấp hơn? **[VIẾT]**
3. Điều này có nghĩa K-Means đang **dự đoán** `Needs_Support` không? Vì sao? **[VIẾT]**
4. Nếu DNU muốn dùng cluster để thiết kế hỗ trợ học tập, cần thận trọng điều gì? **[VIẾT]**


# Optional Challenge — PCA visualization

K-Means đang học trên 5 chiều. PCA dưới đây chỉ dùng để **chiếu xuống 2D cho trực quan hóa**, không thay thế dữ liệu huấn luyện chính.


In [ ]:
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame({
    'PC1': X_2d[:, 0],
    'PC2': X_2d[:, 1],
    'Cluster': final_labels,
})

plt.figure(figsize=(8, 6))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='Cluster', palette='tab10', alpha=0.65)
plt.title('K-Means clusters visualized in 2D with PCA')
plt.show()

print('Explained variance ratio:', pca.explained_variance_ratio_)


### Optional Challenge — nhận xét

- Các cluster có tách biệt rõ trên hình PCA không? **[VIẾT]**
- Vì sao hình 2D này không thể hiện toàn bộ thông tin của dữ liệu 5 chiều? **[VIẾT]**


# Final — Model Reflection Card

## 1. K cuối cùng

`K = [ĐIỀN]`

## 2. Bằng chứng chọn K

[VIẾT: Elbow + Silhouette + khả năng diễn giải]

## 3. Các cluster chính

[VIẾT 2–5 dòng/cluster]

## 4. Quan hệ với `Needs_Support` / `Exam_Score`

[VIẾT]

## 5. Một đề xuất hỗ trợ cho từng nhóm

[VIẾT]

## 6. Hai giới hạn của phân tích

1. [VIẾT]
2. [VIẾT]

## 7. Một câu chốt

> K-Means giúp tôi khám phá ... **[VIẾT]**
